<a href="https://colab.research.google.com/github/JCF-the-creator/Multi-Motorsport-Analytics---Desempenho-e-Estrat-gia/blob/main/F1_World_Analistyc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Importar biblioteca**

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

**Denifir os caminhos**

In [2]:
RAW_PATH = Path('.')
SILVER_PATH = Path('data/silver')
GOLD_PATH = Path('data/gold')

**Carregar todos os CSVs**

In [3]:
datasets = {
    "circuits": pd.read_csv(RAW_PATH / 'Csv' / 'circuits.csv'),
    'constructor_results': pd.read_csv(RAW_PATH / 'Csv/constructor_results.csv'),
    'constructor_standings': pd.read_csv(RAW_PATH / 'Csv/constructor_standings.csv'),
    'constructors': pd.read_csv(RAW_PATH / 'Csv/constructors.csv'),
    'drivers': pd.read_csv(RAW_PATH / 'Csv/drivers.csv'),
    'driver_standings': pd.read_csv(RAW_PATH / 'Csv/driver_standings.csv'),
    'lap_times': pd.read_csv(RAW_PATH / 'Csv/lap_times.csv'),
    'pit_stops': pd.read_csv(RAW_PATH / 'Csv/pit_stops.csv'),
    'qualifying': pd.read_csv(RAW_PATH / 'Csv/qualifying.csv'),
    'races': pd.read_csv(RAW_PATH / 'Csv/races.csv'),
    'results': pd.read_csv(RAW_PATH / 'Csv/results.csv'),
    'seasons': pd.read_csv(RAW_PATH / 'Csv/seasons.csv'),
    'sprint_results': pd.read_csv(RAW_PATH / 'Csv/sprint_results.csv'),
    'status': pd.read_csv(RAW_PATH / 'Csv/status.csv')
  
    
}

**Criando os datasets** | **Verificando duplicatas e Nulo**

In [4]:
for name, df in datasets.items():
  print(name, df.shape)

circuits (77, 9)
constructor_results (12625, 5)
constructor_standings (13391, 7)
constructors (212, 5)
drivers (861, 9)
driver_standings (34863, 7)
lap_times (589081, 6)
pit_stops (11371, 7)
qualifying (10494, 9)
races (1125, 18)
results (26759, 18)
seasons (75, 2)
sprint_results (360, 16)
status (139, 2)


In [5]:
for name, df in datasets.items():
  print(name, df.duplicated().sum())

circuits 0
constructor_results 0
constructor_standings 0
constructors 0
drivers 0
driver_standings 0
lap_times 0
pit_stops 0
qualifying 0
races 0
results 0
seasons 0
sprint_results 0
status 0


Criando dimensões dos pilotos

In [6]:
drivers = datasets["drivers"].copy()

drivers["driver_name"] = drivers["forename"] + " " + drivers["surname"]

dim_driver = drivers[
    [
        "driverId",
        "driverRef",
        "driver_name",
        "dob",
        "nationality"
    ]
]

**Criando Dimensões das equipes**

In [7]:
constructors = datasets["constructors"].copy()

dim_constructor = constructors[
    [
        "constructorId",
        "constructorRef",
        "name",
        "nationality"
    ]
].rename(columns={
    "name": "team_name"
})

**Criando dimensões de circuitos**

In [8]:
circuits = datasets["circuits"].copy()

dim_circuit = circuits[
    [
        "circuitId",
        "circuitRef",
        "name",
        "location",
        "country",
        "lat",
        "lng",
        "alt"
    ]
].rename(columns={
    "name": "circuit_name"
})

**Criando dimensão de corridas**

In [9]:
races = datasets["races"].copy()

dim_race = races[
    [
        "raceId",
        "year",
        "round",
        "circuitId",
        "name",
        "date"
    ]
].rename(columns={
    "name": "race_name"
})

**Criando tabela fato de resultados**

In [10]:
results = datasets["results"].copy()

results["positionOrder"] = pd.to_numeric(
    results["positionOrder"],
    errors="coerce"
)

results["grid"] = pd.to_numeric(
    results["grid"],
    errors="coerce"
)

results["points"] = pd.to_numeric(
    results["points"],
    errors="coerce"
)

fato_resultados = results.copy()

fato_resultados["is_winner"] = fato_resultados["positionOrder"] == 1
fato_resultados["is_podium"] = fato_resultados["positionOrder"] <= 3
fato_resultados["is_top10"] = fato_resultados["positionOrder"] <= 10
fato_resultados["started_from_pole"] = fato_resultados["grid"] == 1
fato_resultados["position_gain"] = fato_resultados["grid"] - fato_resultados["positionOrder"]

**Criando base principla Gold**

In [11]:
f1_gold = fato_resultados.merge(
    dim_driver,
    on="driverId",
    how="left"
)

f1_gold = f1_gold.merge(
    dim_constructor,
    on="constructorId",
    how="left"
)

f1_gold = f1_gold.merge(
    dim_race,
    on="raceId",
    how="left"
)

f1_gold = f1_gold.merge(
    dim_circuit,
    on="circuitId",
    how="left"
)
#ganho medio de posições 

avg_position_gain = (
    f1_gold
    .groupby("driver_name")["position_gain"]
    .mean()
    .reset_index()
    .sort_values(
        "position_gain",
        ascending=False
    )
)
 #Vitoris sem pole position
f1_gold["won_from_non_pole"] = (
    (f1_gold["grid"] > 1)
    &
    (f1_gold["positionOrder"] == 1)
)
  #recuperação de vitórias sem pole position
recovery_wins = (
    f1_gold[
        f1_gold["won_from_non_pole"]
    ]
    .groupby("driver_name")
    .size()
    .reset_index(
        name="wins_without_pole"
    )
    .sort_values(
        "wins_without_pole",
        ascending=False
    )
)


Identificando Abandono (Prova não concluida)

In [12]:
status = datasets["status"].copy()

    # União (JOIN)
f1_gold = f1_gold.merge(
    status,
    on="statusId",
    how="left"
)

    # Criando coluna de abandono (DNF)

f1_gold["is_dnf"] = (
    f1_gold["status"] != "Finished"
)

In [13]:
wins_by_driver = (
    f1_gold[f1_gold["is_winner"]]
    .groupby("driver_name")
    .size()
    .reset_index(name="wins")
    .sort_values("wins", ascending=False)
)

In [14]:
podiums_by_driver = (
    f1_gold[f1_gold["is_podium"]]
    .groupby("driver_name")
    .size()
    .reset_index(name="podiums")
    .sort_values("podiums", ascending=False)
)

In [15]:
points_by_team = (
    f1_gold
    .groupby("team_name")["points"]
    .sum()
    .reset_index()
    .sort_values("points", ascending=False)
)
#taxa de abandono por equipe
dnf_by_team = (
    f1_gold
    .groupby("team_name")
    .agg(
        corridas=("raceId", "count"),
        abandonos=("is_dnf", "sum")
    )
    .reset_index()
)

#Taxa dnf
dnf_by_team["dnf_rate"] = (
    dnf_by_team["abandonos"]
    / dnf_by_team["corridas"]
)


In [16]:
filtered_f1_gold = f1_gold[f1_gold["started_from_pole"]]

pole_conversion_data = {
    "total_poles": filtered_f1_gold["started_from_pole"].sum(),
    "wins_from_pole": filtered_f1_gold["is_winner"].sum()
}
pole_conversion = pd.Series(pole_conversion_data)

pole_conversion["conversion_rate"] = (
    pole_conversion["wins_from_pole"] / pole_conversion["total_poles"]
)
#acima conversão gerl na histori do f1
 #conversão do pole position em vitória po piloto
pole_conversion_driver = (
    f1_gold[
        f1_gold["started_from_pole"]
    ]
    .groupby("driver_name")
    .agg(
        poles=("started_from_pole", "sum"),
        wins=("is_winner", "sum")
    )
    .reset_index()
)

pole_conversion_driver["conversion_rate"] = (
    pole_conversion_driver["wins"]
    /
    pole_conversion_driver["poles"]
)
#taxa de conversão do pole position em vitória por piloto
pole_conversion_driver["conversion_rate"] = (
    pole_conversion_driver["wins"]
    /
    pole_conversion_driver["poles"]
)

#percentual de conversão do pole position em vitória por piloto
pole_conversion_driver["conversion_percent"] = (
    pole_conversion_driver["conversion_rate"]
    * 100
).round(2)


**Criando análise de pit stops**

In [17]:
pit_stops = datasets["pit_stops"].copy()

pit_stops["pit_seconds"] = pit_stops["milliseconds"] / 1000

pit_gold = pit_stops.merge(
    f1_gold[
        [
            "raceId",
            "driverId",
            "driver_name",
            "team_name",
            "year",
            "race_name"
        ]
    ],
    on=["raceId", "driverId"],
    how="left"
)

In [18]:
avg_pit_by_team = (
    pit_gold
    .groupby("team_name")["pit_seconds"]
    .mean()
    .reset_index()
    .sort_values("pit_seconds")
)

In [19]:
pit_count_by_team = (
    pit_gold
    .groupby("team_name")["stop"]
    .count()
    .reset_index(name="pit_stop_count")
    .sort_values("pit_stop_count", ascending=False)
)

**Criando analise de tempos de volta**

In [20]:
lap_times = datasets["lap_times"].copy()

lap_times["lap_seconds"] = lap_times["milliseconds"] / 1000

lap_gold = lap_times.merge(
    f1_gold[
        [
            "raceId",
            "driverId",
            "driver_name",
            "team_name",
            "year",
            "race_name"
        ]
    ],
    on=["raceId", "driverId"],
    how="left"
)

In [21]:
driver_consistency = (
    lap_gold
    .groupby("driver_name")["lap_seconds"]
    .agg(
        avg_lap_time="mean",
        best_lap_time="min",
        lap_consistency="std"
    )
    .reset_index()
    .sort_values("lap_consistency")
)
#consistencia por corrida e melhor e pior volta 
race_consistency = (
    lap_gold
    .groupby(
        [
            "driver_name",
            "year",
            "race_name"
        ]
    )["lap_seconds"]
    .agg(
        media_volta="mean",
        melhor_volta="min",
        pior_volta="max",
        consistencia="std"
    )
    .reset_index()
)
 #consistencia por equipe
team_consistency =(
    lap_gold
   .groupby("team_name")["lap_seconds"]
    .std()
    .reset_index(name="lap_std")
    .sort_values("lap_std")
)
# ranking de media da consistencia por piloto
driver_consistency_rank = (
    race_consistency
    .groupby("driver_name")
    .agg(
        consistencia_media=("consistencia", "mean")
    )
    .reset_index()
    .sort_values(
        "consistencia_media"
    )
)

Criando analise de classifição

In [22]:
qualifying = datasets["qualifying"].copy()

qualifying_gold = qualifying.merge(
    f1_gold[
        [
            "raceId",
            "driverId",
            "driver_name",
            "team_name",
            "year",
            "race_name",
            "positionOrder",
            "is_winner",
            "is_podium"
        ]
    ],
    on=["raceId", "driverId"],
    how="left"
)

In [23]:
qualifying_gold["qualified_top3"] = qualifying_gold["position"] <= 3

qualifying_impact = (
    qualifying_gold
    .groupby("qualified_top3")
    .agg(
        corridas=("raceId", "count"),
        vitorias=("is_winner", "sum"),
        podios=("is_podium", "sum")
    )
    .reset_index()
)

qualifying_impact["taxa_vitoria"] = (
    qualifying_impact["vitorias"] / qualifying_impact["corridas"]
)

qualifying_impact["taxa_podio"] = (
    qualifying_impact["podios"] / qualifying_impact["corridas"]
)

**Criando analise de sprint**

In [24]:
sprint = datasets["sprint_results"].copy()

sprint_gold = sprint.merge(
    dim_driver,
    on="driverId",
    how="left"
).merge(
    dim_constructor,
    on="constructorId",
    how="left"
).merge(
    dim_race,
    on="raceId",
    how="left"
)

In [25]:
sprint_points_team = (
    sprint_gold
    .groupby("team_name")["points"]
    .sum()
    .reset_index()
    .sort_values("points", ascending=False)
)

**Exportar arquivos tratados**

In [26]:
GOLD_PATH.mkdir(parents=True, exist_ok=True)

f1_gold.to_csv(GOLD_PATH / "f1_resultados_gold.csv", index=False)
pit_gold.to_csv(GOLD_PATH / "f1_pit_stops_gold.csv", index=False)
lap_gold.to_csv(GOLD_PATH / "f1_lap_times_gold.csv", index=False)
qualifying_gold.to_csv(GOLD_PATH / "f1_qualifying_gold.csv", index=False)
sprint_gold.to_csv(GOLD_PATH / "f1_sprint_gold.csv", index=False)

In [27]:
import os

os.listdir()

['Csv',
 'data',
 'f1_analytics_gold.zip',
 'F1_World_Analistyc.ipynb',
 'Readme.me']

In [28]:
!zip f1_analytics_gold.zip \
f1_resultados_gold.csv \
f1_pit_stops_gold.csv \
f1_lap_times_gold.csv \
f1_qualifying_gold.csv \
f1_sprint_gold.csv

'zip' is not recognized as an internal or external command,
operable program or batch file.
